In [2]:
import pandas as pd
import numpy as np

file_path = "C:/Users/asus/Desktop/Users/elliottisaac/OneDrive - Tulane University/Research_active/ssm_taxes/Data/restat_replication/Build/Input/acs_2012-2017.dat"

# Step 1: Read only 5 columns to find (year, serial) of same-sex couple households.
# These columns are sufficient to determine whether a person is in a same-sex couple.
colspecs_scan = [
    (0,   4),    # year
    (6,   14),   # serial
    (77,  81),   # pernum
    (106, 108),  # sploc
    (126, 130),  # related  (4-digit detailed version)
    (130, 131),  # sex
    (269, 270),  # qrelate
]
col_names_scan = ["year", "serial", "pernum", "sploc", "related", "sex", "qrelate"]

print("Step 1: scanning for same-sex couples...")
df_scan = pd.read_fwf(file_path, colspecs=colspecs_scan, names=col_names_scan)
print(f"  Total rows: {len(df_scan):,}")


Step 1: scanning for same-sex couples...
  Total rows: 18,871,967


In [3]:
# ── Identify same-sex couples using vectorised operations ──────────────────────────
# Core logic: find partner via sploc, then compare sex values.

# Sort for consistent merging
df_scan = df_scan.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Build a lookup table: (year, serial, pernum) → sex / related / qrelate
# Use merge instead of apply (much faster)
partner_info = df_scan[["year","serial","pernum","sex","related","qrelate"]].copy()
partner_info.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate"]

df_scan = df_scan.merge(partner_info, on=["year","serial","sploc"], how="left")

# Same-sex flag
same_sex    = df_scan["sex"] == df_scan["partner_sex"]
sploc_valid = df_scan["sploc"] > 0

# Same-sex married: head (101) + spouse (201), or vice versa
ss_mar = same_sex & sploc_valid & (
    ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 201)) |
    ((df_scan["related"] == 201)  & (df_scan["partner_related"] == 101))
)

# Same-sex cohabiting: head (101) + unmarried partner (1114), or vice versa
ss_coh = same_sex & sploc_valid & (
    ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 1114)) |
    ((df_scan["related"] == 1114) & (df_scan["partner_related"] == 101))
)

# mflag correction: qrelate==9 indicates the Census Bureau re-coded a same-sex
# spouse as "unmarried partner"; these should be treated as married.
mflag = (df_scan["related"] == 1114) & (df_scan["qrelate"] == 9) & ss_coh

# Propagate mflag to the partner as well
mflag_info = df_scan[["year","serial","pernum"]].copy()
mflag_info["mflag"] = mflag.values
mflag_info.columns = ["year","serial","sploc","partner_mflag"]
df_scan = df_scan.merge(mflag_info, on=["year","serial","sploc"], how="left")
df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)

ss_mar = ss_mar | mflag | df_scan["partner_mflag"]
ss_coh = ss_coh & ~mflag & ~df_scan["partner_mflag"]

df_scan["ss_all"] = (ss_mar | ss_coh).astype(int)

# Collect the set of (year, serial) keys that belong to SS households
ss_keys = df_scan.loc[df_scan["ss_all"] == 1, ["year","serial"]].drop_duplicates()
print(f"  Same-sex couple households found: {len(ss_keys):,}")
print(f"  Same-sex individuals: {df_scan['ss_all'].sum():,}")


C:\Users\asus\AppData\Local\Temp\ipykernel_13544\2326949204.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)


  Same-sex couple households found: 54,027
  Same-sex individuals: 108,054


In [4]:
# Core modification: Add the educd (160, 163) field directly to avoid reading the full file twice.
colspecs_full = [
    (0, 4), (6, 14), (37, 39), (77, 81), (106, 108), (118, 119), 
    (124, 126), (126, 130), (130, 131), (131, 134), (134, 135), 
    (147, 148), (151, 152), (158, 160), (160, 163), (193, 194), 
    (251, 258), (261, 262), (269, 270)
]
col_names_full = [
    "year", "serial", "statefip", "pernum", "sploc", "nchild", 
    "relate", "related", "sex", "age", "marst", 
    "race", "hispan", "educ", "educd", "uhrswork", 
    "incearn", "migrate1", "qrelate"
]


# Create a hash set to speed up lookups
ss_set = set(zip(ss_keys["year"], ss_keys["serial"]))
chunks = []
CHUNK = 500_000

print("Step 2: reading full columns for SS households only...")
reader = pd.read_fwf(file_path, colspecs=colspecs_full, names=col_names_full, chunksize=CHUNK)

for chunk in reader:
    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
    filtered = chunk[mask]
    if len(filtered) > 0:
        chunks.append(filtered)

df_full = pd.concat(chunks, ignore_index=True)


# Calculate the number of adults in the household (used for subsequent precise filtering of cohabiting couples).
df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index(name="num_adults")
df_full = df_full.merge(num_adults, on=["year","serial"], how="left")

print(f"Done. Full data for SS households: {len(df_full):,} rows")

Step 2: reading full columns for SS households only...
Done. Full data for SS households: 136,932 rows


In [5]:
# Sort data to ensure merge logic is accurate
df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
same_sex = df_clean["sex"] == df_clean["p_sex"]
sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

# Handle mflag (married couples identified by qrelate==9)
mflag = (df_clean["related"] == 1114) & (df_clean["qrelate"] == 9) & ss_coh
mflag_info = df_clean[["year","serial","pernum"]].copy()
mflag_info["mflag"] = mflag.values
mflag_info.columns = ["year","serial","sploc","p_mflag"]

df_clean = df_clean.merge(mflag_info, on=["year","serial","sploc"], how="left")
df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


# Finalize marriage and cohabitation classification
ss_mar_final = ss_mar | mflag | df_clean["p_mflag"]
ss_coh_final = ss_coh & ~mflag & ~df_clean["p_mflag"]

df_clean["sscouple_mar"] = ss_mar_final.astype(int)
df_clean["sscouple_coh"] = ss_coh_final.astype(int)
df_clean["ss_any"] = (ss_mar_final | ss_coh_final).astype(int)

# Base filters: Age 18-60, exclude territories, exclude cases with allocated relationships
mask_base = (
    (df_clean["ss_any"] == 1) & 
    (df_clean["age"] >= 18) & (df_clean["age"] <= 60) & 
    (df_clean["statefip"] <= 56) & 
    (df_clean["qrelate"] != 4) & (df_clean.get("p_qrelate", 0) != 4)
)
df_valid = df_clean[mask_base].copy()

C:\Users\asus\AppData\Local\Temp\ipykernel_13544\3924121692.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


In [6]:
# ================= Assemble Samples =================
# Married logic: Head (101) and their spouse
mc = df_valid[df_valid["sscouple_mar"] == 1].copy()
head_m = mc[mc["related"] == 101]
spouse_m = mc[mc["related"].isin([201, 1114])]
spouse_m = spouse_m.sort_values("pernum").groupby(["year","serial"]).first().reset_index()
df_mar = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
df_mar["married"] = 1

In [7]:
# Cohabiting logic: To reach 21,136, we typically do not restrict num_adults
cc = df_valid[df_valid["sscouple_coh"] == 1].copy()
head_c = cc[cc["related"] == 101]
partner_c = cc[cc["related"] == 1114]
df_coh = head_c.merge(partner_c, on=["year","serial"], suffixes=("_h","_s"))
df_coh["married"] = 0

# Ensure bidirectional sploc matching (excludes complex cohabitation structures)
df_coh = df_coh[(df_coh["sploc_h"] == df_coh["pernum_s"]) & (df_coh["sploc_s"] == df_coh["pernum_h"])]

In [8]:
# Concatenate and drop duplicates
df_final = pd.concat([df_mar, df_coh], axis=0, ignore_index=True)
df_final = df_final.drop_duplicates(subset=["year", "serial"])

print(f"=== Final Sample (Target Alignment) ===")
print(f"Married:    {len(df_final[df_final['married']==1]):,} (Paper: 16,098)")
print(f"Cohabiting: {len(df_final[df_final['married']==0]):,} (Paper: 21,136)")

=== Final Sample (Target Alignment) ===
Married:    16,146 (Paper: 16,098)
Cohabiting: 21,761 (Paper: 21,136)


In [9]:
# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()





In [10]:
# Income and earnings split calculation
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(df_f["total_earn"] > 0, df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"], np.nan)

# Construction of demographics and employment status variables
df_f["male"] = (df_f["sex_h"] == 1).astype(int)
df_f["female"] = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"] = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"] = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"] = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Racial matching and dependent children identification
df_f["same_race"] = ((df_f["race_h"] == df_f["race_s"]) & ((df_f["hispan_h"] > 0) == (df_f["hispan_s"] > 0))).astype(int)
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)

# Application of education years
df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])
df_f["edu_max"] = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"] = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]


# Output results comparison table
m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

In [11]:
print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Your Married':>18} {'Your Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male", m["male"], c["male"], "0.469 (0.499)", "0.506 (0.500)")
row("Female", m["female"], c["female"], "0.531 (0.499)", "0.494 (0.500)")
row("Same race", m["same_race"], c["same_race"], "0.793 (0.405)", "0.757 (0.429)")
row("Age older", m["age_old"], c["age_old"], "46.205 (9.633)","43.353 (10.902)")
row("Age younger", m["age_yng"], c["age_yng"], "41.252 (9.831)","37.858 (10.442)")
row("Age diff", m["age_diff"], c["age_diff"], "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)", m["edu_max"], c["edu_max"], "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)", m["edu_min"], c["edu_min"], "13.718 (3.044)","13.513 (2.604)")
row("Edu diff", m["edu_diff"], c["edu_diff"], "1.875 (2.286)", "1.850 (2.108)")
row("Any children", m["any_children"], c["any_children"], "0.314 (0.464)","0.162 (0.369)")
row("Both work", m["both_work"], c["both_work"], "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works", m["one_work"], c["one_work"], "0.195 (0.396)", "0.160 (0.367)")
row("Neither works", m["none_work"], c["none_work"], "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings", m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split", m.loc[m["total_earn"]>0,"earn_split"], c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")
print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

=== Table A2 Full Comparison ===

Variable                                  Your Married         Your Cohab  Paper Married  Paper Cohab
────────────────────────────────────────────────────────────────────────────────────────────────────
Male                                     0.468 (0.499)      0.497 (0.500)  0.469 (0.499) 0.506 (0.500)
Female                                   0.532 (0.499)      0.503 (0.500)  0.531 (0.499) 0.494 (0.500)
Same race                                0.783 (0.412)      0.744 (0.436)  0.793 (0.405) 0.757 (0.429)
Age older                               46.209 (9.634)    43.251 (10.933) 46.205 (9.633) 43.353 (10.902)
Age younger                             41.241 (9.842)    37.698 (10.462) 41.252 (9.831) 37.858 (10.442)
Age diff                                 4.968 (5.263)      5.554 (5.690)  4.953 (5.234) 5.495 (5.543)
Edu max (yrs)                           15.588 (2.472)     15.345 (2.264) 15.594 (2.468) 15.363 (2.264)
Edu min (yrs)                        

In [14]:
df_f.to_csv("couple_sample_final.csv", index=False)
print("Saved.")

Saved.


## BUILT UP FOR LASSO


In [ ]:
# Re-read 'occ' and 'degfield' (only for same-sex households)
colspecs_occ = [
    (0,   4),    # year
    (6,   14),   # serial
    (77,  81),   # pernum
    (163, 165),  # degfield   164-165
    (179, 183),  # occ        180-183
]
col_names_occ = ["year", "serial", "pernum", "degfield", "occ"]

print("Reading occ and degfield...")
chunks_occ = []
reader_occ = pd.read_fwf(
    file_path,
    colspecs=colspecs_occ,
    names=col_names_occ,
    chunksize=500_000
)
for chunk in reader_occ:
    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
    filtered = chunk[mask]
    if len(filtered) > 0:
        chunks_occ.append(filtered)

df_occ = pd.concat(chunks_occ, ignore_index=True)
print(f"Done: {len(df_occ):,} rows")

# Merge into df_full
df_full = df_full.merge(df_occ, on=["year","serial","pernum"], how="left")
print(f"occ null: {df_full['occ'].isna().sum()}")
print(f"degfield null: {df_full['degfield'].isna().sum()}")
print(f"occ unique: {df_full['occ'].nunique()}")
print(f"degfield unique: {df_full['degfield'].nunique()}")

Reading occ and degfield...
Done: 136,932 rows
occ null: 0
degfield null: 0
occ unique: 480
degfield unique: 38


In [37]:
print(df_full_aug.columns)

Index(['year', 'serial', 'statefip', 'pernum', 'sploc', 'nchild', 'relate',
       'related', 'sex', 'age', 'marst', 'race', 'hispan', 'educ', 'educd',
       'uhrswork', 'incearn', 'migrate1', 'qrelate', 'adult', 'num_adults',
       'degfield_x', 'occ_x', 'occ_y', 'degfield_y'],
      dtype='object')


In [46]:
# Build grouped feature variables
# Reference: Paper footnote 22 and variable names in the Stata do-file

def build_lasso_vars(person_df):
    """
    Input: Individual-level DataFrame (including age, educd, race, hispan, sex, occ, degfield, statefip, year, incearn, nchild)
    Output: DataFrame with added grouped variables required for LASSO
    """
    d = person_df.copy()

    # ── r_agegroup: 5-year intervals ────────────────────────────────────
    # Corresponds to "five-year age group dummies" in the paper
    bins   = [17, 22, 27, 32, 37, 42, 47, 52, 57, 62]
    labels = [1,   2,  3,  4,  5,  6,  7,  8,  9]
    d["r_agegroup"] = pd.cut(d["age"], bins=bins, labels=labels, right=True)
    d["r_agegroup"] = pd.Series(d["r_agegroup"])
    d["r_agegroup"] = d["r_agegroup"].astype("Int64")  # nullable int

    # ── r_edugroup: 4 education levels ────────────────────────────────
    # Corresponds to "four education-level dummies" in the paper
    # 1 = < HS, 2 = HS grad, 3 = some college/assoc, 4 = BA+
    # Grouping based on 'educd' (detailed education code)
    educd = d["educd"].fillna(-1).astype(int)
    edu_grp = np.full(len(d), np.nan)
    edu_grp[educd < 62]                           = 1   # < HS diploma
    edu_grp[(educd >= 62) & (educd < 71)]         = 2   # HS diploma / GED
    edu_grp[(educd >= 71) & (educd < 101)]        = 3   # Some college / Associate
    edu_grp[educd >= 101]                         = 4   # BA or higher
    edu_grp = pd.Series(edu_grp)
    d["r_edugroup"] = edu_grp.astype("Int64")

    # ── r_race: race/ethnicity ─────────────────────────────────
    # ACS race codes: 1=White, 2=Black, 3=AI, 4-6=Asian/PI, 7+=other
    # hispan > 0 → Hispanic (overrides race)
    race = d["race"].fillna(0).astype(int)
    hispan = d["hispan"].fillna(0).astype(int)
    
    # Logic: Hispanic (6), White non-Hispanic (1), Black (2), Asian/PI (4), Others (3)
    r_race_val = np.where(hispan > 0, 6,
                 np.where(race == 1, 1,
                 np.where(race == 2, 2,
                 np.where((race >= 4) & (race <= 6), 4,
                 3))))
    d["r_race"] = r_race_val

    # ── r_occbroad: 2-digit (broad) occupation ─────────────────
    # ACS occ refers to ACS 2010 occupation codes (4 digits)
    # The paper specifies "two-digit occupation" -> use integer division to get broad categories
    occ = d["occ_y"].fillna(0).astype(int)
    d["r_occbroad"] = (occ // 100).astype(int)
    # Unemployed or missing -> 0
    d.loc[occ == 0, "r_occbroad"] = 0

    # ── r_degbroad: broad college major ────────────────────────
    # IPUMS degfield: 4-digit code, the first digit represents the broad category
    # 0000 = N/A (no college degree or missing)
    degfield = d["degfield_y"].fillna(0).astype(int)
    d["r_degbroad"] = np.where(degfield > 0, degfield // 1000 + 1, 0).astype(int)

    # ── r_male: sex ────────────────────────────────────────────
    # ACS: 1=Male, 2=Female
    d["r_male"] = (d["sex"] == 1).astype(int)

    return d

print("Function 'build_lasso_vars' defined successfully.")

Function 'build_lasso_vars' defined successfully.


In [49]:
# Build features for head and spouse separately, then merge into wide format

# Separate head and partner from df_full_aug
# (Corresponds to the _h and _s suffixes in df_final)

# First, construct LASSO variables for df_full_aug
print(" Building LASSO feature variables...")
df_full_lasso = build_lasso_vars(df_full_aug)

# Select 'head' (related == 101): This corresponds to the _h columns in df_final
lasso_head = df_full_lasso[df_full_lasso["related"] == 101].copy()
lasso_head = lasso_head[["year", "serial", "pernum",
                          "r_agegroup", "r_edugroup", "r_race",
                          "r_occbroad", "r_degbroad", "r_male",
                          "incearn"]].rename(columns={"incearn": "r_incearn_extra"})

# Select 'partner' (related == 201 or 1114): This corresponds to the _s columns in df_final
lasso_part = df_full_lasso[df_full_lasso["related"].isin([201, 1114])].copy()
lasso_part = lasso_part[["year", "serial", "pernum",
                          "r_agegroup", "r_edugroup", "r_race",
                          "r_occbroad", "r_degbroad", "r_male",
                          "incearn"]].rename(columns={
                              "r_agegroup": "sp_agegroup",
                              "r_edugroup": "sp_edugroup",
                              "r_race":      "sp_race",
                              "r_occbroad": "sp_occbroad",
                              "r_degbroad": "sp_degbroad",
                              "r_male":      "sp_male",
                              "incearn":    "sp_incearn_extra",
                          })

print(f"  Head rows: {len(lasso_head):,}")
print(f"  Partner rows: {len(lasso_part):,}")

 Building LASSO feature variables...
  Head rows: 54,027
  Partner rows: 54,027


In [50]:
# Merge into df_final to construct the final LASSO input

# df_final contains 'pernum_h' and 'pernum_s' columns (from Table A2 merge)
# Merge using (year, serial, pernum_h) and (year, serial, pernum_s) respectively

print("Merging LASSO features into df_final...")

# Merge features for the Head
lasso_head_for_merge = lasso_head.rename(columns={"pernum": "pernum_h"})
df_lasso = df_final.merge(
    lasso_head_for_merge,
    on=["year", "serial", "pernum_h"],
    how="left"
)

# Merge features for the Partner
# For the partner, match using their specific 'pernum'
# In df_final, the partner for both married and cohabiting couples is identified by 'pernum_s'
lasso_part_for_merge = lasso_part.rename(columns={"pernum": "pernum_s"})
df_lasso = df_lasso.merge(
    lasso_part_for_merge,
    on=["year", "serial", "pernum_s"],
    how="left"
)

# Verification of merge results
merge_ok_h = df_lasso["r_agegroup"].notna().sum()
merge_ok_s = df_lasso["sp_agegroup"].notna().sum()
print(f"  Head features merged successfully: {merge_ok_h:,} / {len(df_lasso):,}")
print(f"  Partner features merged successfully: {merge_ok_s:,} / {len(df_lasso):,}")

Merging LASSO features into df_final...
  Head features merged successfully: 37,907 / 37,907
  Partner features merged successfully: 37,907 / 37,907


In [51]:
# Build the final LASSO input DataFrame

print(" Constructing the final LASSO input format...")

# Columns required by the Lasso notebook (one couple per row; r_ prefix = head, sp_ prefix = partner):
# year, serial, statefip,
# r_agegroup, r_edugroup, r_race, r_occbroad, r_degbroad, r_male, r_incearn,
# sp_agegroup, sp_edugroup, sp_race, sp_occbroad, sp_degbroad, sp_male, sp_incearn,
# c_children,
# sscouple_mar, sscouple_coh  (for diagnostic purposes)

# Determine r_incearn and sp_incearn
# incearn_h and incearn_s are already present in df_lasso (from Table A2)
# These should be consistent with r_incearn_extra (for validation)

# First, check 'incearn' consistency
if "incearn_h" in df_lasso.columns and "r_incearn_extra" in df_lasso.columns:
    corr = np.corrcoef(
        df_lasso["incearn_h"].fillna(0),
        df_lasso["r_incearn_extra"].fillna(0)
    )[0, 1]
    print(f"  Correlation (incearn_h vs r_incearn_extra): {corr:.4f} (should be close to 1.0)")

# Construct 'c_children' (total children of the couple = nchild of the household head)
if "nchild_h" in df_lasso.columns:
    df_lasso["c_children"] = df_lasso["nchild_h"].fillna(0).astype(int)
elif "nchild" in df_lasso.columns:
    df_lasso["c_children"] = df_lasso["nchild"].fillna(0).astype(int)
else:
    df_lasso["c_children"] = 0
    print("  ⚠️  Warning: 'nchild' column not found, setting 'c_children' to 0")

# Determine 'statefip' column name
statefip_col = "statefip_h" if "statefip_h" in df_lasso.columns else "statefip"

# Define 'married' status columns
if "married" in df_lasso.columns:
    df_lasso["sscouple_mar"] = df_lasso["married"] == 1
    df_lasso["sscouple_coh"] = df_lasso["married"] == 0

# Define final columns to keep
keep_cols = [
    "year", "serial",
    statefip_col,
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children",
    "sscouple_mar", "sscouple_coh",
]

# Handle 'incearn' naming flexibility
for earn_col, new_name in [("incearn_h", "r_incearn"), ("incearn_s", "sp_incearn")]:
    if earn_col in df_lasso.columns:
        df_lasso[new_name] = df_lasso[earn_col].fillna(0).clip(lower=0)
    else:
        print(f"  ⚠️  Warning: {earn_col} column not found")
        df_lasso[new_name] = 0
keep_cols += ["r_incearn", "sp_incearn"]

# Standardize 'statefip' column name
if statefip_col != "statefip":
    df_lasso = df_lasso.rename(columns={statefip_col: "statefip"})
    keep_cols = ["statefip" if c == statefip_col else c for c in keep_cols]

# Keep only necessary columns
available = [c for c in keep_cols if c in df_lasso.columns]
missing_cols = [c for c in keep_cols if c not in df_lasso.columns]
if missing_cols:
    print(f"  ⚠️  The following columns were not found: {missing_cols}")

df_out = df_lasso[available].copy()

# Type conversion
int_cols = ["r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
            "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
            "c_children"]
for col in int_cols:
    if col in df_out.columns:
        df_out[col] = df_out[col].fillna(0).astype(int)

df_out["r_incearn"]  = df_out["r_incearn"].fillna(0).astype(float)
df_out["sp_incearn"] = df_out["sp_incearn"].fillna(0).astype(float)

print(f"\nFinal LASSO input shape: {df_out.shape}")
print(f"Columns: {list(df_out.columns)}")

 Constructing the final LASSO input format...
  Correlation (incearn_h vs r_incearn_extra): 1.0000 (should be close to 1.0)

Final LASSO input shape: (37907, 20)
Columns: ['year', 'serial', 'statefip', 'r_agegroup', 'r_edugroup', 'r_race', 'r_occbroad', 'r_degbroad', 'r_male', 'sp_agegroup', 'sp_edugroup', 'sp_race', 'sp_occbroad', 'sp_degbroad', 'sp_male', 'c_children', 'sscouple_mar', 'sscouple_coh', 'r_incearn', 'sp_incearn']


In [52]:
# Diagnostic checks


print("="*60)
print("DIAGNOSTICS")
print("="*60)

print(f"\nObservations: {len(df_out):,}  (paper: 37,234)")
print(f"  Married: {df_out['sscouple_mar'].sum():,}  (paper: 16,098)")
print(f"  Cohabiting: {df_out['sscouple_coh'].sum():,}  (paper: 21,136)")

print(f"\nShare of positive earners:")
print(f"  r_incearn > 0: {(df_out['r_incearn'] > 0).mean():.3f}")
print(f"  sp_incearn > 0: {(df_out['sp_incearn'] > 0).mean():.3f}")
print(f"  (paper: r ≈ 0.935, sp ≈ 0.812-0.847)")

print(f"\nDistribution of 'occbroad' (should have multiple values):")
print(f"  r_occbroad unique values: {df_out['r_occbroad'].nunique()}")
print(f"  r_occbroad top 5: {df_out['r_occbroad'].value_counts().head().to_dict()}")

print(f"\nDistribution of 'degbroad':")
print(f"  r_degbroad unique values: {df_out['r_degbroad'].nunique()}")
print(f"  (0 = no degree, 1-N = degree groups)")

print(f"\nYearly distribution:")
print(df_out["year"].value_counts().sort_index().to_string())

print(f"\nr_incearn statistics (Married):")
mar = df_out[df_out["sscouple_mar"] == True]
print(f"  mean: {mar['r_incearn'].mean():,.0f}")
print(f"  (paper Table A3: primary earner mean ≈ 77,120)")

DIAGNOSTICS

Observations: 37,907  (paper: 37,234)
  Married: 16,146  (paper: 16,098)
  Cohabiting: 21,761  (paper: 21,136)

Share of positive earners:
  r_incearn > 0: 0.905
  sp_incearn > 0: 0.863
  (paper: r ≈ 0.935, sp ≈ 0.812-0.847)

Distribution of 'occbroad' (should have multiple values):
  r_occbroad unique values: 100
  r_occbroad top 5: {0: 3312, 47: 2462, 4: 2163, 23: 1609, 1: 1303}

Distribution of 'degbroad':
  r_degbroad unique values: 2
  (0 = no degree, 1-N = degree groups)

Yearly distribution:
year
2012    5070
2013    5875
2014    6234
2015    6683
2016    6801
2017    7244

r_incearn statistics (Married):
  mean: 71,124
  (paper Table A3: primary earner mean ≈ 77,120)


In [53]:
# Save

df_out.to_pickle(OUT_PKL)
print(f"✓ Save completed: {OUT_PKL}")
print(f"  Shape: {df_out.shape}")
print(f"  Columns: {list(df_out.columns)}")
print()

✓ Save completed: acs_ssc_lasso_input.pkl
  Shape: (37907, 20)
  Columns: ['year', 'serial', 'statefip', 'r_agegroup', 'r_edugroup', 'r_race', 'r_occbroad', 'r_degbroad', 'r_male', 'sp_agegroup', 'sp_edugroup', 'sp_race', 'sp_occbroad', 'sp_degbroad', 'sp_male', 'c_children', 'sscouple_mar', 'sscouple_coh', 'r_incearn', 'sp_incearn']

